In [ ]:
# COMP263 Deep Learning Project
# Airline Sentiment Detection (LSTM baseline)  Group 7 
# Student: Atiqa Sheriff (300507716), Krishan Singh (300936234)
# Ajmal Afzalzada (301413451), Nihethan Mahendran (300971960 )

import os
import re
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, GlobalMaxPooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2


# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paths — works on any machine as long as project structure is kept intact
NOTEBOOK_DIR = Path(os.path.abspath(""))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "Tweets.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

# GloVe is in the project root folder
GLOVE_PATH = PROJECT_ROOT / "glove.6B.100d.txt"

FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load data
print(f"Using data file: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)

print("\nData overview")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

# Keep only needed columns
df = df[["text", "airline_sentiment"]].dropna()

print("\nRaw class distribution")
print(df["airline_sentiment"].value_counts())

# Save class distribution plot
plt.figure(figsize=(6, 4))
order = df["airline_sentiment"].value_counts().index
sns.countplot(x="airline_sentiment", data=df, order=order, color="skyblue")
plt.title("Class Distribution - Airline Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=300)
plt.show()

# Text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[!?]{2,}", " multiexclaim ", text)
    text = re.sub(r"!", " exclaim ", text)
    text = re.sub(r"\?", " question ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].copy()

print("\nSample cleaned text")
print(df[["text", "clean_text"]].head(5))

# Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["airline_sentiment"])
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\nLabel mapping")
print(label_mapping)

# Split data (Train / Val / Test = 70 / 15 / 15)
X = df["clean_text"].values
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("\nData split")
print("Train size:", len(X_train))
print("Val size  :", len(X_val))
print("Test size :", len(X_test))

# Tokenize and pad
max_words = 20000
max_len = 55

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

print("\nTokenized shapes")
print("X_train_pad:", X_train_pad.shape)
print("X_val_pad  :", X_val_pad.shape)
print("X_test_pad :", X_test_pad.shape)

# Load GloVe pre-trained embeddings
# glove.6B.100d.txt is included in the project root folder
embedding_dim = 100

glove_index = {}
with open(GLOVE_PATH, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_index[word] = vector

embedding_matrix = np.zeros((max_words, embedding_dim))
for word, idx in tokenizer.word_index.items():
    if idx < max_words:
        vec = glove_index.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec

print(f"GloVe loaded. Coverage: {np.count_nonzero(embedding_matrix.sum(axis=1))} / {max_words} words")

# Class weights
classes = np.unique(y_train)
class_weights_array = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, class_weights_array)}

print("\nClass weights")
print(class_weights)

class_weights[1] = class_weights[1] * 1.4
print("Adjusted weights (neutral boosted):", class_weights)

# Build model
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=max_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
    GlobalMaxPooling1D(),
    Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
    Dropout(0.4),
    Dense(64, activation="relu", kernel_regularizer=l2(1e-4)),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.build(input_shape=(None, max_len))

print("\nModel summary")
model.summary()

with open(OUTPUT_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))

# Train model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

lr_reducer = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop, lr_reducer],
    verbose=1
)

print("\nTraining completed!")

# Training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves_lstm.png", dpi=300)
plt.show()

# Evaluate model
y_prob = model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

prec_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("\nTest metrics")
print(f"Accuracy            : {acc:.4f}")
print(f"Precision (Macro)   : {prec_macro:.4f}")
print(f"Recall (Macro)      : {rec_macro:.4f}")
print(f"F1-score (Macro)    : {f1_macro:.4f}")
print(f"Precision (Weighted): {prec_weighted:.4f}")
print(f"Recall (Weighted)   : {rec_weighted:.4f}")
print(f"F1-score (Weighted) : {f1_weighted:.4f}")

report = classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0)
print("\nClassification report")
print(report)

with open(OUTPUT_DIR / "classification_report_lstm.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - LSTM")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_lstm.png", dpi=300)
plt.show()

# Save results table
metrics_df = pd.DataFrame([{
    "accuracy": acc,
    "precision_macro": prec_macro,
    "recall_macro": rec_macro,
    "f1_macro": f1_macro,
    "precision_weighted": prec_weighted,
    "recall_weighted": rec_weighted,
    "f1_weighted": f1_weighted
}])

metrics_path = OUTPUT_DIR / "metrics_lstm.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nFinal results")
print(metrics_df)

# ROC-AUC curves
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_prob_all = model.predict(X_test_pad, verbose=0)

plt.figure(figsize=(8, 6))
colors = ["#e74c3c", "#3498db", "#2ecc71"]

for i, cls_name in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_all[:, i])
    auc_val = roc_auc_score(y_test_bin[:, i], y_prob_all[:, i])
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f"{cls_name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - BiLSTM Airline Sentiment")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_curves_lstm.png", dpi=300)
plt.show()

macro_roc_auc = roc_auc_score(y_test_bin, y_prob_all, average="macro")
print(f"\nMacro ROC-AUC: {macro_roc_auc:.4f}")

# Save model, tokenizer, and label encoder
model_path = MODEL_DIR / "lstm_airline_sentiment.h5"
tokenizer_path = MODEL_DIR / "tokenizer.pkl"
label_encoder_path = MODEL_DIR / "label_encoder.pkl"

model.save(model_path)

with open(tokenizer_path, "wb") as f:
    pickle.dump(tokenizer, f)

with open(label_encoder_path, "wb") as f:
    pickle.dump(le, f)

print("\nSaved artifacts")
print("Model        :", model_path)
print("Tokenizer    :", tokenizer_path)
print("Label Encoder:", label_encoder_path)
print("Metrics CSV  :", metrics_path)